# nb_05 — Interactive scatter + light-curve explorer

Goal (see `docs/SPEC_V01.md`, rough plan step 6): a reusable function — click a point in any
scatter plot (CMD, period-amplitude, ...), see that object's light curve on the right, with
toggles to fold on a period and to show/hide flux errors. Implemented once in
`src/visualization/lc_explorer.py` (`interactive_scatter_lc`), not copy-pasted per notebook —
this notebook is a demo of it against `dia_object_lc_hq` (nb_01's base HQ sample with nb_02's
stats, nb_03's magnitudes, and nb_04's periods all merged into it in place).

**Tech stack: `holoviews` + `bokeh` + `panel`, not `plotly`.** nb-v01 built this on
`plotly.graph_objects.FigureWidget` + `ipywidgets`, which needed `anywidget` installed
separately (not in the RSP kernel by default) and, once, a browser reload before the widget's
frontend model registered. `holoviews`/`bokeh`/`panel` all ship in the RSP `lsst-scipipe`
kernel already — matching the stack RSP's own interactive-plot tutorials use
(`notebooks/tutorials/DP2/300_Science_demos/312_Interactive_plots`) — so there's nothing extra
to install, and one less thing to go wrong for workshop attendees who've already seen that
stack in an earlier tutorial.

**HQ sample is too large to materialize whole for a live click demo.** nb-v01's version pulled
its entire subset (7,036 objects, ~90 MB) into memory once, since `interactive_scatter_lc`'s
`lc_df` has to already be materialized for a click to feel instant. The HQ sample is ~399k
objects — section 1 below picks a slice (one partition or a cone search, same
`select_slice` helper as nb_02-04) instead of the whole thing.
</cell id="intro">

In [ ]:
import sys
from pathlib import Path

# src/ isn't pip-installed on RSP (pyproject.toml is pinned to Python 3.14, newer than RSP's
# 3.13.9 kernel — see README) — import it straight from the repo checkout instead.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import lsdb
from datapaths import Datapaths
from dataio import select_slice
from visualization import interactive_scatter_lc

dp = Datapaths()
hq_path = dp["dp2_subset"] / "dia_object_lc_hq"
hq_cat = lsdb.open_catalog(hq_path)
print(hq_cat.npartitions, "partitions")
print(sorted(hq_cat.columns))

## 1. Pick a slice of the HQ sample to explore

Same `select_slice` pattern as nb_02-04, but here the slice *is* the working dataset for the
rest of the notebook, not just something to glance at — `interactive_scatter_lc`'s `lc_df`
needs to already be materialized (fetching a light curve per click has to be fast, no lazy
per-click `.compute()`), and the whole ~399k-object HQ sample is too big to pull into memory
for that. One partition or one cone search gives a few dozen to a few hundred objects — small
enough to hold in memory and reuse as both the light-curve source and the scatter-plot source
below.
</cell id="3112e303">

In [ ]:
SLICE_MODE = "partition"  # "partition" or "cone_search"
PARTITION_INDEX = 200
CONE_RA, CONE_DEC, CONE_RADIUS_ARCSEC = 150.0, 2.0, 1800  # ~0.5 deg

slice_cat = select_slice(
    hq_cat,
    mode=SLICE_MODE,
    partition_index=PARTITION_INDEX,
    ra=CONE_RA,
    dec=CONE_DEC,
    radius_arcsec=CONE_RADIUS_ARCSEC,
)
full_df = slice_cat.compute()
print(f"{SLICE_MODE}: {full_df.shape}")

## 2. Demo: color-magnitude diagram (nb_03)

`g-r` vs `r`, colored by `max_reliability` (nb_03's per-object max real/bogus score), folded
on `best_period_days` (nb_04's single-band period) when available. Click a point on the left
to load its light curve on the right; toggle folded vs. unfolded and flux errors on/off.
</cell id="c2bbe626">

In [3]:
full_df['diaObjectForcedSource'].columns

['band',
 'coord_dec',
 'coord_ra',
 'diff_PixelFlags_nodataCenter',
 'invalidPsfFlag',
 'midpointMjdTai',
 'pixelFlags_bad',
 'pixelFlags_cr',
 'pixelFlags_crCenter',
 'pixelFlags_edge',
 'pixelFlags_interpolated',
 'pixelFlags_interpolatedCenter',
 'pixelFlags_nodata',
 'pixelFlags_saturated',
 'pixelFlags_saturatedCenter',
 'pixelFlags_suspect',
 'pixelFlags_suspectCenter',
 'psfDiffFlux',
 'psfDiffFlux_flag',
 'psfDiffFluxErr',
 'psfFlux',
 'psfFlux_flag',
 'psfFluxErr',
 'psfMag',
 'psfMagErr',
 'visit']

In [ ]:
cmd_df = full_df.assign(gr=full_df["g_mag_median"] - full_df["r_mag_median"]).dropna(subset=["gr", "r_mag_median"])
print(cmd_df.shape)

interactive_scatter_lc(
    scatter_df=cmd_df,
    x_col="gr",
    y_col="r_mag_median",
    lc_df=full_df,
    color_col="max_reliability",
    period_col="best_period_days",
    scatter_title="CMD: g-r vs r",
    mag_col="psfDiffFlux", magerr_col="psfDiffFluxErr", nested_col="diaObjectForcedSource"
)

## 3. Demo: period-amplitude diagram (nb_04 x nb_02)

`multiband_period_days` (log-scaled, nb_04's higher-coverage period) vs. `r_amp_p90p10`
(nb_02's robust amplitude), colored by `duration_days` and folded on the same
`multiband_period_days`. This is exactly the combination the spec's rough plan step 5 asks for
as a static plot — here it's interactive instead, in the same function used for the CMD above.
</cell id="ffdb54e6">

In [ ]:
pa_df = full_df.dropna(subset=["multiband_period_days", "r_amp_p90p10"])
print(pa_df.shape)

interactive_scatter_lc(
    scatter_df=pa_df,
    x_col="multiband_period_days",
    y_col="r_amp_p90p10",
    lc_df=full_df,
    color_col="duration_days",
    period_col="multiband_period_days",
    scatter_title="period-amplitude: multiband period vs r-band amplitude",
    mag_col="psfDiffFlux", magerr_col="psfDiffFluxErr", nested_col="diaObjectForcedSource",
    x_log=True,
)

## Next

`src/visualization/lc_explorer.py`'s `interactive_scatter_lc` is the reusable piece; this
notebook is just two example calls against real HQ-sample output. Open questions, not resolved
here:

- **Not tested outside JupyterLab on RSP.** The spec explicitly flags Jupyter-vs-VSCode/IDE
  differences for interactive widgets — unverified either way here.
- **Categorical `color_col` legend — resolved by the bokeh/holoviews rewrite.** The old
  plotly version couldn't show a per-category legend on a single trace (category rode along in
  hover text instead); `holoviews`'s native categorical coloring shows a real legend, no
  workaround needed. Verified against a categorical column during this rewrite (see
  `docs/changelog.md`), though neither demo above happens to use one (HQ's `max_reliability`/
  `duration_days` are both numeric).
- **`lc_df` must already be materialized** — no support for handing it a lazy `lsdb.Catalog`
  and computing per click. Section 1 now picks a slice of the HQ sample specifically because of
  this; would need rethinking (streaming fetch per click?) before pointing this at the full
  ~399k-object sample directly.
- **Doesn't add its own quality filtering.** All of nb_04's caveats about `*_period_power` not
  being a calibrated false-alarm probability still apply when browsing by period here — clicking
  a high-power point doesn't mean the fold is real.
- **Single light-curve panel, not one subplot per band** — matches the spec's wording ("the LC
  plotting panel", singular), but multi-band light curves with very different flux scales can
  be hard to read overlaid; worth revisiting if that turns out to matter in practice.
</cell id="c7abba23">